In [ ]:
import pandas as pd
import numpy as np
import os


pi = 3.14159265359

maxval=1e9
minval=1e-9

In [ ]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from models.mlp_encoder_model_nonquantized import *

In [ ]:
#model=CreateModel_Slim_SoftQuantizer((16,16,2), initial_thresholds=[400, 1000, 2000], threshold_offset=0.0)
model=CreateModel_Slim((16,16,2))
model.summary()

In [ ]:
# get best weights file
pitch = '50x12P5'
batch_size = 5000
fingerprint = '34c2da80'
timeslices = 2
files = os.listdir('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))

vlosses = [float(f.split("-v")[1].split(".hdf5")[0]) for f in files]
bestfile = files[np.argmin(vlosses)]
model.load_weights('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)

print('Best model: {}'.format(bestfile))

In [ ]:
# load in the test set
test_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/{}t/TFR_val_contained_digitize-manual_mlp-SLIM'.format(timeslices),
    quantize = False # False for soft quantizer and manually quantized inputs
)

In [ ]:
# predicts test data
p_test = model.predict(test_generator)

complete_truth = None
for _, y in test_generator:
    if complete_truth is None:
        complete_truth = y
    else:
        complete_truth = np.concatenate((complete_truth, y), axis=0)

# creates df with all predicted values and matrix elements - 4 predictions, all 10 unique matrix elements
df = pd.DataFrame(p_test,columns=['x','y','cotB'])

# stores all true values in same matrix as xtrue, ytrue, etc.
df['xtrue'] = complete_truth[:,0]
df['ytrue'] = complete_truth[:,1]
df['cotBtrue'] = complete_truth[:,2]

# calculates residuals for x, y, cotA, cotB
df['residualsX'] = df['xtrue'] - df['x']
df['residualsY'] = df['ytrue'] - df['y']
df['residualsB'] = df['cotBtrue'] - df['cotB']

# stores results as parquet
df.to_parquet("/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5/{}t-mlp_SLIM-manual_input_digitization-vars.parquet".format(timeslices))